# openSIP full workflow: from data pairs to graph

This notebook demonstrates the complete openSIP workflow:

1. collect or construct geotagged sound/image metadata,
2. build audio-subclip to image candidate pairs,
3. score embedding, semantic, and affective-quality similarity,
4. finalize the pair dataframe schema,
5. export a bipartite graph where audio sub-clips and images are nodes.

The notebook runs offline by default with a small synthetic dataset. Cells for live API collection and heavy model inference are included but guarded by flags.

## 0. Setup

The default installation includes data collection, ImageBind similarity, SoundAQnet, outdoor filtering, and graph export:

```bash
pip install openSIP
```

PlacePulse image AQ is the only optional extra. Install it only when using the built-in Hugging Face PlacePulse checkpoints:

```bash
pip install "openSIP[placepulse]"
```

In [1]:
from __future__ import annotations

import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from opensip import AudioImagePairDataset, GeoTaggedData

WORK_DIR = Path("opensip_demo_outputs")
MEDIA_DIR = WORK_DIR / "media"
WORK_DIR.mkdir(exist_ok=True)
MEDIA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

RUN_LIVE_COLLECTION = True
RUN_IMAGEBIND = True
RUN_SOUNDAQNET = True
RUN_PLACEPULSE = True

/Users/xiaohaoyang/Documents/open_sound_image_pair/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Optional live data collection

Set `RUN_LIVE_COLLECTION = True` in the setup cell to run this section with your own API keys. The calls are designed to accumulate data across sources and remove duplicates.

In [ ]:
if RUN_LIVE_COLLECTION:
    BBOX = (-83.080, 42.345, -83.060, 42.355)  # replace with your study area
    apd = AudioImagePairDataset()

    # Audio collection can be called repeatedly. openSIP accumulates and deduplicates rows.
    apd.get_sound_from_location(bbox=BBOX, source="aporee", slice_duration=30)
    apd.get_sound_from_location(bbox=BBOX, source="freesound", key=os.environ.get("FREESOUND_KEY"), slice_duration=30)

    # SVI collection can also accumulate. Google requires an API key.
    apd.get_svi_from_locations(source="google", key=os.environ.get("GOOGLE_STREETVIEW_KEY"))
    # apd.get_svi_from_locations(source="mapillary", key=os.environ.get("MAPILLARY_KEY"))

    # Flickr photo collection can remove selfie/person-focused photos with the built-in YuNet model.
    apd.get_photo_from_location(key=os.environ.get("FLICKR_KEY"), remove_selfies=True)

    # Optional outdoor filtering. Prefer Overture for larger areas; OSM often times out.
    # apd.filter_outdoor(data="both", building_source="overture", bbox=BBOX)
else:
    print("Live collection skipped. The next section builds an offline demo dataset.")

## 2. Offline demo dataset

The synthetic dataset has two audio clips and four images: two street-view images and two Flickr photos. Each audio sub-clip connects to multiple images at the same location, which is the structure we want to see in the final graph.

In [ ]:
def make_demo_image(path: Path, color: tuple[int, int, int], size=(320, 220)) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.new("RGB", size, color=color).save(path)
    return str(path)


img_svi_1 = make_demo_image(MEDIA_DIR / "svi_main_street.jpg", (80, 130, 180))
img_svi_2 = make_demo_image(MEDIA_DIR / "svi_park_edge.jpg", (60, 150, 90))
img_photo_1 = make_demo_image(MEDIA_DIR / "flickr_sidewalk.jpg", (180, 130, 80))
img_photo_2 = make_demo_image(MEDIA_DIR / "flickr_fountain.jpg", (70, 170, 190))

gtd = GeoTaggedData(locations=[[-83.0730, 42.3480], [-83.0710, 42.3490]])
gtd.audios = {
    "loc_id": [0, 1],
    "id": ["aud-traffic", "aud-park"],
    "data": ["https://example.org/audio/traffic.mp3", "https://example.org/audio/park.mp3"],
    "path": ["", ""],
    "slice": [[0, 30_000], [0, 30_000]],
}
gtd.audio_metadata = pd.DataFrame(
    [
        {
            "id": "aud-traffic",
            "created": "2024-05-18T09:15:00",
            "latitude": 42.3480,
            "longitude": -83.0730,
        },
        {
            "id": "aud-park",
            "created": "2024-05-19T15:30:00",
            "latitude": 42.3490,
            "longitude": -83.0710,
        },
    ]
)
gtd.svi_metadata = pd.DataFrame(
    [
        {
            "loc_id": 0,
            "id": "svi-main-street",
            "image_url": img_svi_1,
            "captured_at": "2024-05-18-10",
            "latitude": 42.3481,
            "longitude": -83.0727,
        },
        {
            "loc_id": 1,
            "id": "svi-park-edge",
            "image_url": img_svi_2,
            "captured_at": "2024-05-1-1",
            "latitude": 42.3491,
            "longitude": -83.0713,
        },
    ]
)
gtd.photo_metadata = pd.DataFrame(
    [
        {
            "loc_id": 0,
            "id": "flickr-sidewalk",
            "url": img_photo_1,
            "datetaken": "2024-05-18T09:45:00",
            "latitude": 42.3482,
            "longitude": -83.0729,
        },
        {
            "loc_id": 1,
            "id": "flickr-fountain",
            "url": img_photo_2,
            "datetaken": "2024-05-19T16:05:00",
            "latitude": 42.3492,
            "longitude": -83.0712,
        },
    ]
)

apd = AudioImagePairDataset(gtd)
apd.gtd.audio_metadata

## 3. Build sound-image pair candidates

`build_pairs()` builds one image source at a time. To combine street-view and Flickr photos, build both tables and concatenate them into `apd.pairs`.

In [ ]:
svi_pairs = apd.build_pairs(image_source="svi", time_window=60).copy()
photo_pairs = apd.build_pairs(image_source="photo", time_window=60).copy()
apd.pairs = pd.concat([svi_pairs, photo_pairs], ignore_index=True)

apd.pairs[[
    "loc_id", "audio_id", "clip_index", "audio_source", "image_id",
    "image_source", "time_diff_days", "similarity"
]]

## 4. Embedding similarity

For real media, call `compute_embedding_similarity()`. The fast default lets ImageBind sample three 2-second audio windows; `audio_strategy="exhaustive"` splits the full segment into consecutive 2-second windows.

The offline demo assigns deterministic example scores so the notebook stays fast and runnable without model weights.

In [ ]:
if RUN_IMAGEBIND:
    apd.compute_embedding_similarity(
        output_dir=WORK_DIR / "embedding_media",
        checkpoint_path=WORK_DIR / "embedding_similarity.jsonl",
        audio_strategy="sample",
        clips_per_audio=3,
        max_image_edge=1024,
    )
else:
    demo_scores = {
        ("aud-traffic", "svi-main-street"): 0.42,
        ("aud-traffic", "flickr-sidewalk"): 0.36,
        ("aud-park", "svi-park-edge"): 0.31,
        ("aud-park", "flickr-fountain"): 0.39,
    }
    apd.pairs["similarity"] = [
        demo_scores.get((row.audio_id, row.image_id), np.nan)
        for row in apd.pairs.itertuples(index=False)
    ]
    apd.pairs["n_sub_clips"] = 3

apd.pairs[["audio_id", "image_id", "similarity", "n_sub_clips"]]

## 5. Semantic event-rank similarity

The semantic layer compares ranked sound-event labels. In production, openSIP can infer audio events with SoundAQnet and image events with zero-shot ImageBind over the same fixed event list. Here we pass event rankings directly so the cell is offline.

In [ ]:
audio_events = {
    ("aud-traffic", 0): ["Traffic", "Vehicle", "Human_sounds", "Noise", "Speech"],
    ("aud-park", 0): ["Water", "Bird", "Natural_sounds", "Human_sounds", "Wind"],
}
image_events = {
    "svi-main-street": ["Traffic", "Vehicle", "Human_sounds", "Noise", "Speech"],
    "flickr-sidewalk": ["Human_sounds", "Traffic", "Speech", "Vehicle", "Noise"],
    "svi-park-edge": ["Bird", "Natural_sounds", "Wind", "Water", "Human_sounds"],
    "flickr-fountain": ["Water", "Natural_sounds", "Human_sounds", "Bird", "Wind"],
}

if RUN_SOUNDAQNET:
    apd.compute_semantic_similarity(
        infer_audio_events=True,
        infer_image_events=True,
        image_event_backend="zeroshot",
        output_dir=WORK_DIR / "semantic_media",
        image_event_checkpoint_path=WORK_DIR / "image_event_ranks.jsonl",
        top_k=5,
    )
else:
    apd.compute_semantic_similarity(
        audio_events=audio_events,
        image_events=image_events,
        infer_audio_events=False,
        infer_image_events=True,
        top_k=5,
    )

apd.pairs[["audio_id", "image_id", "semantic_ndcg_at_k", "semantic_mrr", "semantic_similarity"]]

## 6. Affective-quality similarity

There are different AQ similarity routes, so the final schema keeps them distinct:

- `direct`: compare supplied or SoundAQnet-predicted AQ vectors,
- `event_rank`: approximate AQ from top ranked sound events and an event-to-AQ table,
- `placepulse`: compare SoundAQnet sound AQ to PlacePulse-style image `lively`/`boring` scores mapped to `vibrant`/`monotonous`.

The offline demo shows direct AQ and event-rank AQ. PlacePulse is included as an optional guarded call and requires `pip install "openSIP[placepulse]"` when using the built-in checkpoints.

In [ ]:
aq_dimensions = ["pleasant", "eventful", "chaotic", "vibrant", "uneventful", "calm", "annoying", "monotonous"]

audio_aq = {
    ("aud-traffic", 0): {
        "pleasant": 2.0, "eventful": 7.5, "chaotic": 6.5, "vibrant": 5.0,
        "uneventful": 2.0, "calm": 2.5, "annoying": 6.0, "monotonous": 3.0,
    },
    ("aud-park", 0): {
        "pleasant": 7.0, "eventful": 4.0, "chaotic": 2.0, "vibrant": 5.5,
        "uneventful": 4.5, "calm": 7.5, "annoying": 1.5, "monotonous": 2.0,
    },
}
image_aq = {
    "svi-main-street": {
        "pleasant": 2.5, "eventful": 7.0, "chaotic": 6.0, "vibrant": 5.5,
        "uneventful": 2.5, "calm": 3.0, "annoying": 5.5, "monotonous": 3.5,
    },
    "flickr-sidewalk": {
        "pleasant": 3.0, "eventful": 6.5, "chaotic": 5.5, "vibrant": 5.0,
        "uneventful": 3.0, "calm": 3.5, "annoying": 5.0, "monotonous": 4.0,
    },
    "svi-park-edge": {
        "pleasant": 6.5, "eventful": 4.0, "chaotic": 2.5, "vibrant": 5.0,
        "uneventful": 4.5, "calm": 7.0, "annoying": 2.0, "monotonous": 2.5,
    },
    "flickr-fountain": {
        "pleasant": 7.5, "eventful": 4.5, "chaotic": 2.0, "vibrant": 6.0,
        "uneventful": 4.0, "calm": 7.5, "annoying": 1.0, "monotonous": 1.5,
    },
}

apd.compute_aq_similarity(audio_aq=audio_aq, image_aq=image_aq, aq_dimensions=aq_dimensions)
direct_aq_similarity = apd.pairs["aq_similarity"].copy()
apd.pairs["aq_similarity_direct"] = direct_aq_similarity
apd.pairs[["audio_id", "image_id", "aq_similarity_type", "aq_rmse", "aq_pearson", "aq_similarity_direct"]]

In [ ]:
event_aq_scores = {
    "Traffic": {"pleasant": 2, "eventful": 8, "chaotic": 7, "vibrant": 5, "uneventful": 2, "calm": 2, "annoying": 7, "monotonous": 4},
    "Vehicle": {"pleasant": 2, "eventful": 7, "chaotic": 6, "vibrant": 5, "uneventful": 2, "calm": 2, "annoying": 6, "monotonous": 4},
    "Noise": {"pleasant": 1, "eventful": 8, "chaotic": 8, "vibrant": 3, "uneventful": 1, "calm": 1, "annoying": 8, "monotonous": 5},
    "Human_sounds": {"pleasant": 4, "eventful": 6, "chaotic": 4, "vibrant": 6, "uneventful": 3, "calm": 4, "annoying": 4, "monotonous": 3},
    "Speech": {"pleasant": 4, "eventful": 5, "chaotic": 3, "vibrant": 5, "uneventful": 3, "calm": 4, "annoying": 3, "monotonous": 3},
    "Water": {"pleasant": 8, "eventful": 4, "chaotic": 2, "vibrant": 6, "uneventful": 5, "calm": 8, "annoying": 1, "monotonous": 2},
    "Bird": {"pleasant": 8, "eventful": 5, "chaotic": 2, "vibrant": 6, "uneventful": 4, "calm": 7, "annoying": 1, "monotonous": 2},
    "Natural_sounds": {"pleasant": 8, "eventful": 4, "chaotic": 2, "vibrant": 5, "uneventful": 5, "calm": 8, "annoying": 1, "monotonous": 2},
    "Wind": {"pleasant": 6, "eventful": 4, "chaotic": 3, "vibrant": 4, "uneventful": 5, "calm": 6, "annoying": 2, "monotonous": 3},
}

apd.compute_aq_similarity(
    audio_events=audio_events,
    image_events=image_events,
    event_aq_scores=event_aq_scores,
    aq_dimensions=aq_dimensions,
    infer_audio_aq=False,
    infer_audio_events=False,
    infer_image_events=True,
    top_k=5,
)
apd.pairs["aq_similarity_event_rank"] = apd.pairs["aq_similarity"]
apd.pairs["aq_similarity_direct"] = direct_aq_similarity

if RUN_PLACEPULSE:
    apd.compute_aq_similarity(
        audio_aq=audio_aq,
        infer_image_placepulse=True,
        placepulse_checkpoint_path=WORK_DIR / "placepulse.jsonl",
        placepulse_panorama_strategy="auto",
        aq_dimensions=["vibrant", "monotonous"],
        output_dir=WORK_DIR / "placepulse_media",
    )
    apd.pairs["aq_similarity_placepulse"] = apd.pairs["aq_similarity"]

apd.pairs[[
    "audio_id", "image_id", "aq_similarity_type", "aq_similarity",
    "aq_similarity_direct", "aq_similarity_event_rank"
]]

## 7. Filter and finalize the pair dataframe

Filtering keeps the best images per audio sub-clip. `finalize_pairs()` adds the canonical schema and computes `distance_m` from audio/image coordinates.

In [ ]:
apd.filter_pairs_by_similarity(min_similarity=0.30, min_sub_clips=1, top_k_per_audio=2)
final_pairs = apd.finalize_pairs()

core_cols = [
    "pair_id", "audio_node_id", "image_node_id", "audio_id", "image_id",
    "image_source", "distance_m", "time_diff_days", "embedding_similarity",
    "semantic_similarity", "aq_similarity", "aq_similarity_type",
    "aq_similarity_direct", "aq_similarity_event_rank", "selected",
]
final_pairs[core_cols]

In [ ]:
final_csv = apd.export_final_dataframe(WORK_DIR / "final_pairs.csv")
metadata_csv = apd.export(WORK_DIR / "pairs_metadata", data=False)
print("Final dataframe:", final_csv)
print("Pair metadata:", metadata_csv)

## 8. Build and export the sound-image graph

The graph is bipartite:

- audio sub-clips are `audio_clip` nodes,
- street-view/Flickr images are `image` nodes,
- pair rows are edges with `distance_m`, `time_diff_days`, and similarity attributes.

In [ ]:
graph = apd.to_graph(final_pairs)
print("nodes:", graph.number_of_nodes())
print("edges:", graph.number_of_edges())

first_edge = next(iter(graph.edges(data=True)))
first_edge

In [ ]:
graphml_path = apd.export_graph(WORK_DIR / "sound_image_pair_graph.graphml", pairs=final_pairs)
gexf_path = apd.export_graph(WORK_DIR / "sound_image_pair_graph.gexf", pairs=final_pairs)
print(graphml_path)
print(gexf_path)

In [ ]:
try:
    fig, ax = apd.plot_pair_graph(
        final_pairs,
        color_edges_by="embedding_similarity",
        width_edges_by="embedding_similarity",
        label_nodes=True,
        label_edges=True,
        figsize=(11, 5),
    )
    fig.savefig(WORK_DIR / "sound_image_pair_graph.png", dpi=180, bbox_inches="tight")
except ImportError as exc:
    print("Graph plotting skipped because matplotlib is unavailable or misconfigured:", exc)

## 9. Save and resume workflow state

`save()` snapshots the underlying `GeoTaggedData` state and the current pair table. This is useful after expensive collection or model-inference steps.

In [ ]:
state_dir = WORK_DIR / "state"
apd.save(state_dir)

apd_loaded = AudioImagePairDataset.load(state_dir)
apd_loaded.finalize_pairs()[core_cols].head()

## 10. Summary

At the end of the workflow you have:

- `final_pairs.csv`: one row per audio-subclip/image edge,
- `sound_image_pair_graph.graphml` and `.gexf`: graph files for Gephi, NetworkX, or GIS-adjacent analysis,
- `sound_image_pair_graph.png`: a quick visual check of the connections,
- `state/`: a resumable openSIP workflow snapshot.

The graph edge attributes preserve distance, time difference, embedding similarity, semantic similarity, and typed AQ similarity columns.